# Meta Ads — Gold Star Schema

Builds dimensions, `fact_ad_performance_daily`, reporting table `rpt_meta_ad_performance_daily`, and SQL view `vw_meta_ad_performance`.

**Budget rule:** Silver `*_amount` = paise/100. Gold exposes as `*_inr` (do not divide again).

**Incremental:** default `FULL_REFRESH=False` merges new fact/report dates; history kept.


In [ ]:
silver_schema = "silver"
gold_schema = "gold"
run_optimize = True

# Incremental (default): merge new dates; do not wipe history
FULL_REFRESH = False
INCREMENTAL_LOOKBACK_DAYS = 2
INCREMENTAL_MAX_DAYS = 14


In [ ]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")


# Incremental helpers (controls: FULL_REFRESH / LOOKBACK / MAX_DAYS in parameters cell)
from delta.tables import DeltaTable

def _table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(1).collect()
        return True
    except Exception:
        return False

def _path_is_delta(path: str) -> bool:
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except Exception:
        return False

def _max_date(table_or_path: str, date_col: str, is_path: bool = False):
    try:
        df = spark.read.format("delta").load(table_or_path) if is_path else spark.table(table_or_path)
        return df.agg(F.max(F.col(date_col)).alias("m")).collect()[0]["m"]
    except Exception:
        return None

def filter_by_watermark(df, date_col: str, target: str, is_path: bool = False):
    if FULL_REFRESH:
        print(f"[FULL_REFRESH] no watermark filter: {target}")
        return df
    exists = _path_is_delta(target) if is_path else _table_exists(target)
    if not exists:
        print(f"[INCR] target missing → first load: {target}")
        return df
    wm = _max_date(target, date_col, is_path=is_path)
    if wm is None:
        print(f"[INCR] empty watermark → full batch: {target}")
        return df
    out = df.filter(F.col(date_col).isNotNull() & (F.col(date_col) >= F.date_sub(F.lit(wm), int(INCREMENTAL_LOOKBACK_DAYS))))
    st = out.agg(F.min(date_col).alias("mn"), F.max(date_col).alias("mx"), F.count(F.lit(1)).alias("n")).collect()[0]
    print(f"[INCR] {target} wm={wm} lookback={INCREMENTAL_LOOKBACK_DAYS}d rows={st['n']} range={st['mn']}..{st['mx']}")
    if st["n"] and st["mn"] is not None and st["mx"] is not None:
        span = (st["mx"] - st["mn"]).days
        if span > int(INCREMENTAL_MAX_DAYS):
            raise ValueError(
                f"Incremental batch for {target} spans {span} days (> {INCREMENTAL_MAX_DAYS}). "
                "Refusing large backfill. Use daily/2-day bronze, or set FULL_REFRESH=True intentionally."
            )
    return out

def merge_or_overwrite_table(df, target: str, keys, partition_cols=None, stamp_col="silver_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _table_exists(target):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.saveAsTable(target)
        mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "MERGE"
    print(f"[OK] {target} ({mode}) total={spark.table(target).count():,}")

def merge_or_overwrite_path(df, path: str, keys, partition_cols=None, stamp_col="_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _path_is_delta(path):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("mergeSchema", "true")
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.save(path)
        mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forPath(spark, path).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "MERGE"
    print(f"[OK] {path} ({mode}) total={spark.read.format('delta').load(path).count():,}")


def write_gold(df, table, keys, partition_cols=None, date_col=None):
    target = f"{gold_schema}.{table}"
    batch = df
    if date_col:
        batch = filter_by_watermark(batch, date_col, target)
        if len(batch.take(1)) == 0:
            print(f"[SKIP] {target}: no new incremental rows")
            return
    merge_or_overwrite_table(batch, target, keys, partition_cols=partition_cols, stamp_col="gold_processed_at")
    if run_optimize:
        spark.sql(f"OPTIMIZE {target}")


def add_sk(df, business_key, sk_name):
    w = Window.orderBy(business_key)
    return df.withColumn(sk_name, F.row_number().over(w))


## Dimensions

In [ ]:
campaigns = spark.table(f"{silver_schema}.meta_campaigns")
adsets = spark.table(f"{silver_schema}.meta_adsets")
ads = spark.table(f"{silver_schema}.meta_ads")
insights = spark.table(f"{silver_schema}.meta_ad_insights_daily")
actions = spark.table(f"{silver_schema}.meta_insight_actions")

dim_account = add_sk(
    campaigns.select("account_id", "account_name", "platform", "tenant_id", "connector_id").dropDuplicates(["account_id"]),
    "account_id",
    "account_sk",
).select("account_sk", "account_id", "account_name", "platform", "tenant_id", "connector_id")
write_gold(dim_account, "dim_account", keys=["account_id"])

# Budgets already paise/100 in Silver *_amount → expose as *_inr
dim_campaign = (
    add_sk(campaigns, "campaign_id", "campaign_sk")
    .join(dim_account.select("account_id", "account_sk"), "account_id", "left")
    .select(
        "campaign_sk",
        "campaign_id",
        "account_sk",
        "campaign_name",
        "objective",
        "status",
        "configured_status",
        "effective_status",
        "buying_type",
        "bid_strategy",
        F.col("daily_budget_amount").alias("daily_budget_inr"),
        F.col("lifetime_budget_amount").alias("lifetime_budget_inr"),
        F.col("budget_remaining_amount").alias("budget_remaining_inr"),
        "start_time",
        "stop_time",
        "created_time",
        "updated_time",
    )
)
write_gold(dim_campaign, "dim_campaign", keys=["campaign_id"])

dim_adset = (
    add_sk(adsets, "adset_id", "adset_sk")
    .join(dim_campaign.select("campaign_id", "campaign_sk"), "campaign_id", "left")
    .select(
        "adset_sk",
        "adset_id",
        "campaign_sk",
        "adset_name",
        "status",
        "optimization_goal",
        "billing_event",
        "bid_strategy",
        F.col("daily_budget_amount").alias("daily_budget_inr"),
        F.col("lifetime_budget_amount").alias("lifetime_budget_inr"),
        "age_min",
        "age_max",
        "geo_countries",
        "created_time",
        "updated_time",
    )
)
write_gold(dim_adset, "dim_adset", keys=["adset_id"])

dim_ad = (
    add_sk(ads, "ad_id", "ad_sk")
    .join(dim_adset.select("adset_id", "adset_sk"), "adset_id", "left")
    .join(dim_campaign.select("campaign_id", "campaign_sk"), "campaign_id", "left")
    .select(
        "ad_sk",
        "ad_id",
        "adset_sk",
        "campaign_sk",
        "ad_name",
        "status",
        "effective_status",
        "creative_id",
        "preview_shareable_link",
        "created_time",
        "updated_time",
    )
)
write_gold(dim_ad, "dim_ad", keys=["ad_id"])

dim_date = (
    insights.select(F.to_date("date_start").alias("full_date"))
    .where(F.col("full_date").isNotNull())
    .dropDuplicates()
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("year"),
        F.month("full_date").alias("month"),
        F.dayofmonth("full_date").alias("day"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        (F.dayofweek("full_date").isin(1, 7)).cast("int").alias("is_weekend"),
    )
)
write_gold(dim_date, "dim_date", keys=["date_key"])


## Fact + reporting table

In [ ]:
action_map = {
    "lead": "leads",
    "link_click": "link_clicks",
    "post_engagement": "post_engagements",
    "video_view": "video_views",
}

ad_actions = (
    actions.where((F.col("entity_type") == "ad") & F.col("action_type").isin(list(action_map.keys())))
    .withColumn("metric", F.col("action_type"))
    .groupBy("entity_id", "date_start", "metric")
    .agg(F.sum("action_value").alias("action_value"))
    .groupBy("entity_id", "date_start")
    .pivot("metric", list(action_map.keys()))
    .sum("action_value")
    .withColumnRenamed("entity_id", "ad_id")
    .withColumnRenamed("lead", "leads")
    .withColumnRenamed("link_click", "link_clicks")
    .withColumnRenamed("post_engagement", "post_engagements")
    .withColumnRenamed("video_view", "video_views")
)

for c in ["leads", "link_clicks", "post_engagements", "video_views"]:
    if c not in ad_actions.columns:
        ad_actions = ad_actions.withColumn(c, F.lit(0.0))
    else:
        ad_actions = ad_actions.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))

# Incremental: only new/recent insight dates into fact (history kept via MERGE)
insights_incr = insights.withColumn("_full_date", F.to_date("date_start"))
insights_incr = filter_by_watermark(insights_incr, "_full_date", f"{gold_schema}.rpt_meta_ad_performance_daily")

fact = (
    insights_incr.alias("i")
    .join(ad_actions.alias("a"), ["ad_id", "date_start"], "left")
    .join(dim_ad.select("ad_id", "ad_sk", "adset_sk", "campaign_sk"), "ad_id", "left")
    .join(dim_account.select("account_id", "account_sk"), "account_id", "left")
    .select(
        F.date_format(F.to_date("date_start"), "yyyyMMdd").cast("int").alias("date_key"),
        "account_sk",
        "campaign_sk",
        "adset_sk",
        "ad_sk",
        F.col("impressions").cast("double"),
        F.col("reach").cast("double"),
        F.col("frequency").cast("double"),
        F.col("clicks").cast("double"),
        F.col("unique_clicks").cast("double"),
        F.col("inline_link_clicks").cast("double"),
        F.col("spend").cast("double").alias("spend_inr"),
        F.col("cpc").cast("double"),
        F.col("cpm").cast("double"),
        F.col("ctr").cast("double"),
        F.coalesce(F.col("leads"), F.lit(0.0)).alias("leads"),
        F.coalesce(F.col("link_clicks"), F.lit(0.0)).alias("link_clicks"),
        F.coalesce(F.col("post_engagements"), F.lit(0.0)).alias("post_engagements"),
        F.coalesce(F.col("video_views"), F.lit(0.0)).alias("video_views"),
        F.col("source_batch_id"),
    )
    .where(
        F.col("date_key").isNotNull()
        & F.col("account_sk").isNotNull()
        & F.col("campaign_sk").isNotNull()
        & F.col("adset_sk").isNotNull()
        & F.col("ad_sk").isNotNull()
    )
    .withColumn(
        "cpl_inr",
        F.when(F.col("leads") > 0, F.col("spend_inr") / F.col("leads")).otherwise(F.lit(None).cast("double")),
    )
)

fact = fact.select(
    "date_key",
    "account_sk",
    "campaign_sk",
    "adset_sk",
    "ad_sk",
    "impressions",
    "reach",
    "frequency",
    "clicks",
    "unique_clicks",
    "inline_link_clicks",
    "spend_inr",
    "cpc",
    "cpm",
    "ctr",
    "leads",
    "link_clicks",
    "post_engagements",
    "video_views",
    "cpl_inr",
    "source_batch_id",
)
write_gold(fact, "fact_ad_performance_daily", keys=["ad_sk", "date_key"], partition_cols=["date_key"])


In [ ]:
# Single reporting table (denormalized)
rpt = (
    spark.table(f"{gold_schema}.fact_ad_performance_daily").alias("f")
    .join(spark.table(f"{gold_schema}.dim_date").alias("d"), "date_key")
    .join(spark.table(f"{gold_schema}.dim_account").alias("a"), "account_sk")
    .join(spark.table(f"{gold_schema}.dim_campaign").alias("c"), "campaign_sk")
    .join(spark.table(f"{gold_schema}.dim_adset").alias("s"), "adset_sk")
    .join(spark.table(f"{gold_schema}.dim_ad").alias("ad"), "ad_sk")
    .select(
        F.col("d.full_date"),
        F.col("d.year"),
        F.col("d.month"),
        F.col("d.month_name"),
        F.col("d.day_name"),
        F.col("a.account_id"),
        F.col("a.account_name"),
        F.col("a.platform"),
        F.col("c.campaign_id"),
        F.col("c.campaign_name"),
        F.col("c.objective").alias("campaign_objective"),
        F.col("c.status").alias("campaign_status"),
        F.col("c.daily_budget_inr").alias("campaign_daily_budget_inr"),
        F.col("c.lifetime_budget_inr").alias("campaign_lifetime_budget_inr"),
        F.col("c.budget_remaining_inr").alias("campaign_budget_remaining_inr"),
        F.col("s.adset_id"),
        F.col("s.adset_name"),
        F.col("s.status").alias("adset_status"),
        F.col("s.optimization_goal"),
        F.col("s.daily_budget_inr").alias("adset_daily_budget_inr"),
        F.col("s.lifetime_budget_inr").alias("adset_lifetime_budget_inr"),
        F.col("ad.ad_id"),
        F.col("ad.ad_name"),
        F.col("ad.status").alias("ad_status"),
        F.col("ad.effective_status").alias("ad_effective_status"),
        F.col("ad.creative_id"),
        F.col("f.impressions"),
        F.col("f.reach"),
        F.col("f.frequency"),
        F.col("f.clicks"),
        F.col("f.unique_clicks"),
        F.col("f.inline_link_clicks"),
        F.col("f.spend_inr"),
        F.col("f.cpc"),
        F.col("f.cpm"),
        F.col("f.ctr"),
        F.col("f.leads"),
        F.col("f.link_clicks"),
        F.col("f.post_engagements"),
        F.col("f.video_views"),
        F.col("f.cpl_inr"),
        F.col("f.source_batch_id"),
    )
)
write_gold(rpt, "rpt_meta_ad_performance_daily", keys=["account_id", "campaign_id", "adset_id", "ad_id", "full_date"], partition_cols=["full_date"], date_col="full_date")


## SQL view + validation

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_schema}.vw_meta_ad_performance AS
SELECT
    f.fact_sk,
    d.full_date,
    d.year,
    d.month,
    d.month_name,
    d.day_of_week,
    d.day_name,
    a.account_id,
    a.account_name,
    a.platform,
    c.campaign_id,
    c.campaign_name,
    c.objective AS campaign_objective,
    c.status AS campaign_status,
    c.daily_budget_inr AS campaign_daily_budget_inr,
    c.lifetime_budget_inr AS campaign_lifetime_budget_inr,
    c.budget_remaining_inr AS campaign_budget_remaining_inr,
    s.adset_id,
    s.adset_name,
    s.status AS adset_status,
    s.optimization_goal,
    s.daily_budget_inr AS adset_daily_budget_inr,
    s.lifetime_budget_inr AS adset_lifetime_budget_inr,
    ad.ad_id,
    ad.ad_name,
    ad.status AS ad_status,
    ad.effective_status AS ad_effective_status,
    ad.creative_id,
    f.impressions,
    f.reach,
    f.frequency,
    f.clicks,
    f.unique_clicks,
    f.inline_link_clicks,
    f.spend_inr,
    f.cpc,
    f.cpm,
    f.ctr,
    f.leads,
    f.link_clicks,
    f.post_engagements,
    f.video_views,
    f.cpl_inr,
    f.source_batch_id,
    f.gold_processed_at
FROM {gold_schema}.fact_ad_performance_daily f
INNER JOIN {gold_schema}.dim_date d ON f.date_key = d.date_key
INNER JOIN {gold_schema}.dim_account a ON f.account_sk = a.account_sk
INNER JOIN {gold_schema}.dim_campaign c ON f.campaign_sk = c.campaign_sk
INNER JOIN {gold_schema}.dim_adset s ON f.adset_sk = s.adset_sk
INNER JOIN {gold_schema}.dim_ad ad ON f.ad_sk = ad.ad_sk
""")

print("View created: gold.vw_meta_ad_performance")
print("View rows:", spark.table(f"{gold_schema}.vw_meta_ad_performance").count())

checks = spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {gold_schema}.fact_ad_performance_daily) AS fact_rows,
  (SELECT COUNT(*) FROM {gold_schema}.rpt_meta_ad_performance_daily) AS rpt_rows,
  (SELECT SUM(spend_inr) FROM {gold_schema}.fact_ad_performance_daily) AS spend_inr,
  (SELECT SUM(leads) FROM {gold_schema}.fact_ad_performance_daily) AS leads,
  (SELECT COUNT(*) FROM {gold_schema}.fact_ad_performance_daily f
   LEFT ANTI JOIN {gold_schema}.dim_ad d ON f.ad_sk = d.ad_sk) AS orphan_ads
""").collect()[0]
print(dict(checks.asDict()))
assert checks["fact_rows"] == checks["rpt_rows"]
assert checks["orphan_ads"] == 0
print("Gold validation passed")